# 02b - Data preparation

**Aim:** prepare non-redundant positive and negative datasets for signal-peptide prediction.

We cluster the sequences from 02a, retain one representative per cluster, and recover their metadata. We assign training/benchmarking splits and five training folds, then export one FASTA and one TSV per class. The TSVs record each protein's split and fold.

**Materials**
- `02b-DataPreparation-2026.pdf` (slides 4, 6 and 27-29)


## 0. Load the collected datasets

We load the positive and negative FASTA files from 02a. The TSV metadata will be read after clustering.


### 0.1 Imports and paths


In [1]:
import csv
import random
from pathlib import Path

input_dir = Path("../data/collected")
work_dir = Path("../data/mmseqs2")
output_dir = Path("../data/prepared")


### 0.2 Sequence reader

We keep full sequences in a dictionary keyed by `accession`.


In [2]:
def read_fasta(fasta_path):
    sequences = {}
    with open(fasta_path, encoding="utf-8") as fasta_in:
        for line in fasta_in:
            line = line.strip()
            if line.startswith(">"):
                accession = line[1:].split()[0]
                sequences[accession] = ""
            elif line:
                sequences[accession] += line
    return sequences


### 0.3 Load the collected proteins


In [3]:
positive_sequences = read_fasta(input_dir / "positive_dataset.fasta")
negative_sequences = read_fasta(input_dir / "negative_dataset.fasta")

print("Positive proteins:", len(positive_sequences))
print("Negative proteins:", len(negative_sequences))


Positive proteins: 2957
Negative proteins: 20974


## 1. Reduce redundancy

We cluster positives and negatives separately with MMseqs2, using connected components, 30% minimum identity and 40% coverage of both sequences (slides 27 and 29).

Optional installation with Conda in a Linux notebook environment. Uncomment the cell once, then comment it again for normal runs.


In [4]:
# %conda install -c conda-forge -c bioconda mmseqs2 -y

### 1.1 Cluster the sequences

We use `easy-cluster` with the slide parameters. MMseqs2 writes its representative FASTA, cluster assignments and temporary files to `data/mmseqs2/`, separately from the final datasets.


In [5]:
work_dir.mkdir(parents=True, exist_ok=True)

!mmseqs easy-cluster {input_dir}/positive_dataset.fasta {work_dir}/positive {work_dir}/tmp_positive --min-seq-id 0.3 -c 0.4 --cov-mode 0 --cluster-mode 1
!mmseqs easy-cluster {input_dir}/negative_dataset.fasta {work_dir}/negative {work_dir}/tmp_negative --min-seq-id 0.3 -c 0.4 --cov-mode 0 --cluster-mode 1


Create directory ../data/mmseqs2/tmp_positive
easy-cluster ../data/collected/positive_dataset.fasta ../data/mmseqs2/positive ../data/mmseqs2/tmp_positive --min-seq-id 0.3 -c 0.4 --cov-mode 0 --cluster-mode 1 

MMseqs Version:                     	18.8cc5c
Substitution matrix                 	aa:blosum62.out,nucl:nucleotide.out
Seed substitution matrix            	aa:VTML80.out,nucl:nucleotide.out
Sensitivity                         	4
k-mer length                        	0
Target search mode                  	0
k-score                             	seq:2147483647,prof:2147483647
Alphabet size                       	aa:21,nucl:5
Max sequence length                 	65535
Max results per query               	20
Split database                      	0
Split mode                          	2
Split memory limit                  	0
Coverage threshold                  	0.4
Coverage mode                       	0
Compositional bias                  	1
Compositional bias scale            	1
Diagona

### 1.2 Select representatives

The `*_rep_seq.fasta` files contain one representative per cluster (slide 28).


In [6]:
positive_representatives = read_fasta(work_dir / "positive_rep_seq.fasta")
negative_representatives = read_fasta(work_dir / "negative_rep_seq.fasta")

print("Positive representatives:", len(positive_representatives))
print("Negative representatives:", len(negative_representatives))


Positive representatives: 1102
Negative representatives: 9082


### 1.3 Retain representative metadata

We read the TSV files from 02a and retain the representatives' metadata in memory, preserving the original fields (slide 29). We export the final TSVs after assigning splits and folds.


In [7]:
def read_metadata(tsv_path):
    metadata = {}
    with open(tsv_path, encoding="utf-8", newline="") as tsv_in:
        for row in csv.DictReader(tsv_in, delimiter="\t"):
            row["length"] = int(row["length"])
            if "cleavage_pos" in row:
                row["cleavage_pos"] = int(row["cleavage_pos"])
            if "has_tm_in_90" in row:
                row["has_tm_in_90"] = row["has_tm_in_90"] == "True"
            metadata[row["accession"]] = row
    return metadata



In [8]:
positive_metadata = read_metadata(input_dir / "positive_dataset.tsv")
negative_metadata = read_metadata(input_dir / "negative_dataset.tsv")

positive_metadata = {accession: positive_metadata[accession] for accession in positive_representatives}
negative_metadata = {accession: negative_metadata[accession] for accession in negative_representatives}


## 2. Split training and benchmarking datasets

We randomly assign 80% of the representatives in each class to training and the remaining 20% to benchmarking (slides 4, 6 and 29). We use seed 13 to make the split reproducible and round the training count down to a whole number.

In [9]:
random_seed = 13
training_fraction = 0.8
rng = random.Random(random_seed)


def split_representatives(representatives, training_fraction, rng):
    accessions = list(representatives)
    rng.shuffle(accessions)
    n_training = int(training_fraction * len(accessions))
    return accessions[:n_training], accessions[n_training:]


positive_training, positive_benchmarking = split_representatives(
    positive_representatives, training_fraction, rng
)
negative_training, negative_benchmarking = split_representatives(
    negative_representatives, training_fraction, rng
)

print("Training positives:", len(positive_training))
print("Training negatives:", len(negative_training))
print("Benchmarking positives:", len(positive_benchmarking))
print("Benchmarking negatives:", len(negative_benchmarking))

Training positives: 881
Training negatives: 7265
Benchmarking positives: 221
Benchmarking negatives: 1817


## 3. Assign five cross-validation folds

We distribute each class across five training subsets, keeping the positive/negative ratio as close as possible with whole proteins (slide 29). The training lists were already shuffled in step 2, so we assign their accessions to folds in turn. Benchmarking proteins are not included.


In [10]:
n_folds = 5

positive_folds = [positive_training[i::n_folds] for i in range(n_folds)]
negative_folds = [negative_training[i::n_folds] for i in range(n_folds)]

for i in range(n_folds):
    print("Fold:", i + 1)
    print("Positive proteins:", len(positive_folds[i]))
    print("Negative proteins:", len(negative_folds[i]))


Fold: 1
Positive proteins: 177
Negative proteins: 1453
Fold: 2
Positive proteins: 176
Negative proteins: 1453
Fold: 3
Positive proteins: 176
Negative proteins: 1453
Fold: 4
Positive proteins: 176
Negative proteins: 1453
Fold: 5
Positive proteins: 176
Negative proteins: 1453


### 3.1 Export the prepared datasets

We save `positive_dataset` and `negative_dataset` as FASTA and TSV in `data/prepared/`, with matching accession order. The TSVs retain the original metadata and add `split` (`training` or `benchmarking`) and `cv_fold` (1–5 for training, blank for benchmarking).

The fold assignments identify which proteins to use for validation in each cross-validation round; the other four folds form its training set.


In [11]:
def write_prepared_dataset(sequences, metadata, folds, output_prefix):
    fold_by_accession = {}
    for i, fold in enumerate(folds, start=1):
        for accession in fold:
            fold_by_accession[accession] = i

    columns = list(next(iter(metadata.values()))) + ["split", "cv_fold"]
    with open(output_prefix.with_suffix(".tsv"), "w", encoding="utf-8", newline="") as tsv_out, \
         open(output_prefix.with_suffix(".fasta"), "w", encoding="utf-8") as fasta_out:
        writer = csv.DictWriter(tsv_out, fieldnames=columns, delimiter="\t")
        writer.writeheader()
        for accession, sequence in sequences.items():
            row = metadata[accession].copy()
            row["split"] = "training" if accession in fold_by_accession else "benchmarking"
            row["cv_fold"] = fold_by_accession.get(accession, "")
            writer.writerow(row)
            print(f">{accession}", sequence, sep="\n", file=fasta_out)


output_dir.mkdir(parents=True, exist_ok=True)
write_prepared_dataset(positive_representatives, positive_metadata, positive_folds,
                       output_dir / "positive_dataset")
write_prepared_dataset(negative_representatives, negative_metadata, negative_folds,
                       output_dir / "negative_dataset")
